Answer the problem in the context of topics covered in SDS. You are **not allowed** to seek or receive help from anyone to solve the problem. To keep the problem at reasonable difficulty, you are also **not allowed** to use LLMs such as ChatGPT, Gemini, Claude and Llama. You can **only use built-in libraries** of Apache Spark. For example, you cannot use Spark packages.

# Problem 2 [25 pts]

In this problem, assume your data is from 1 million (mobile) IoT devices that send the following information every 10 ms:

* Device ID
* City ADM3 code
* Temperature
* Humidity
* Wind speed
* Air quality

## Problem 2a [5 pts]

Design an algorithm or method that will randomly sample 5% of the devices in real-time and in a reproducible manner without preselecting or storing the list of devices to sample. You may answer this problem by means of a narrative (essay), pseudocode, flowchart, and/or illustration. Make sure that you explicitly write down the necessary parameter values.

### Added solution for Problem 2a

## Problem 2a [5 pts] — Reproducible 5% real-time device sampling

Use **hash-based deterministic sampling**.

For every incoming record, compute a stable hash of `device_id` with a fixed seed/salt. Convert the hash to a number in `[0, 1)`. Keep the record if the number is below `0.05`.

This samples devices, not individual records. Therefore, once a device is selected, all records from that device are selected, and once a device is not selected, all records from that device are rejected. No list of sampled devices needs to be stored.

**Parameters**

- Sampling probability: `p = 0.05`
- Fixed seed/salt: for example, `seed = "SDS_EXAM_2026"`
- Hash function: MurmurHash3, SHA-256, or Spark `xxhash64`
- Selection rule: keep device if `hash(device_id, seed) mod 1,000,000 < 50,000`

**Pseudocode**

```text
p = 0.05
M = 1,000,000
threshold = p * M = 50,000
seed = "SDS_EXAM_2026"

for each incoming record r:
    h = stable_hash(seed || r.device_id)
    bucket = h mod M
    if bucket < threshold:
        emit r
    else:
        discard r
```

**Why this works**

The hash behaves like a random number generator, but because it is deterministic and uses a fixed seed, the same `device_id` always receives the same decision. The method is reproducible, streaming-friendly, and uses `O(1)` memory.


In [2]:
# Optional Spark example for Problem 2a
# Assumes a streaming/static DataFrame df with column device_id.
from pyspark.sql import functions as F

p = 0.05
M = 1_000_000
threshold = int(p * M)
seed_col = F.lit("SDS_EXAM_2026")

sampled_df = (
    df
    .withColumn("sample_bucket", F.pmod(F.xxhash64(seed_col, F.col("device_id")), F.lit(M)))
    .filter(F.col("sample_bucket") < threshold)
    .drop("sample_bucket")
)


AssertionError: 

## Problem 2b [5 pts]

Design an algorithm or method that will select, in real-time, data from a list of 1,000 devices without storing the list of devices itself. Minimize the memory requirement as much as possible, allowing a false positive rate of 1%, but there should be no false negative. You may answer this problem by means of a narrative (essay), pseudocode, flowchart, and/or illustration. Make sure that you explicitly write down the necessary parameter values.

### Added solution for Problem 2b

## Problem 2b [5 pts] — Select data from 1,000 devices without storing the list

Use a **Bloom filter**.

A Bloom filter stores membership information compactly. It can say that a device is “possibly in the target list” or “definitely not in the target list.” It allows false positives, but it has **no false negatives**, which matches the requirement.

Target list size: `n = 1,000` devices  
False positive rate: `f = 0.01`

Bloom filter size:

$$
m = \frac{n \ln f}{(\ln 2)^2}
$$

With `n = 1000` and `f = 0.01`:


$$
m \approx -\frac{1000 \ln(0.01)}{(\ln 2)^2} \approx 9586 \text{ bits}
$$


Number of hash functions:

$$
k = \frac{m}{n}\ln 2\approx 6.64 \approx 7
$$

**Parameters**

- Bloom filter bit array size: `m = 9,586 bits` (about 1.2 KB)
- Number of hash functions: `k = 7`
- Expected inserted devices: `n = 1,000`
- False positive rate: `1%`

**Pseudocode**

```text
m = 9586
k = 7
B = bit array of length m initialized to 0

# setup phase
for each target device d in target_device_list:
    for i = 1 to k:
        pos = hash_i(d) mod m
        B[pos] = 1

# real-time filtering phase
for each incoming record r:
    accept = true
    for i = 1 to k:
        pos = hash_i(r.device_id) mod m
        if B[pos] == 0:
            accept = false
            break
    if accept:
        emit r
    else:
        discard r
```

**Why this works**

If a device was inserted into the Bloom filter, all its `k` bit positions are guaranteed to be 1, so there are no false negatives. A non-target device may accidentally have all `k` positions set by other devices, producing a false positive, but the probability is controlled at about 1%.


In [ ]:
# Optional pure-Python Bloom filter helper for Problem 2b.
# This is for explanation/testing. In Spark streaming, the resulting bit array can be broadcast.
import math
import hashlib

n = 1_000
false_positive_rate = 0.01
m = math.ceil(-(n * math.log(false_positive_rate)) / (math.log(2) ** 2))
k = round((m / n) * math.log(2))
print(m, k)  # expected: about 9586 bits and 7 hashes

class BloomFilter:
    def __init__(self, m, k, seed="SDS_EXAM_2026"):
        self.m = m
        self.k = k
        self.seed = seed
        self.bits = bytearray(m)

    def _hashes(self, value):
        value = str(value)
        for i in range(self.k):
            text = f"{self.seed}|{i}|{value}".encode("utf-8")
            digest = hashlib.sha256(text).hexdigest()
            yield int(digest, 16) % self.m

    def add(self, value):
        for pos in self._hashes(value):
            self.bits[pos] = 1

    def might_contain(self, value):
        return all(self.bits[pos] == 1 for pos in self._hashes(value))


## Problem 2c [5 pts]

Design an algorithm or method that will randomly select 100 data points, with equal probability, among all data points seen so far. You may answer this problem by means of a narrative (essay), pseudocode, flowchart, and/or illustration. Make sure that you explicitly write down the necessary parameter values.

### Added solution for Problem 2c

## Problem 2c [5 pts] — Randomly select 100 data points among all points seen so far

Use **reservoir sampling** with reservoir size `k = 100`.

Reservoir sampling maintains a fixed-size random sample from a stream whose final size is unknown. After seeing `t` records, every record among the first `t` records has equal probability `100/t` of being in the reservoir.

**Parameters**

- Reservoir size: `k = 100`
- Counter: `t`, number of data points seen so far
- Random number generator seed: fixed if reproducibility is required

**Pseudocode**

```text
k = 100
R = empty reservoir
t = 0

for each incoming data point x:
    t = t + 1

    if t <= k:
        append x to R
    else:
        j = random integer from 1 to t
        if j <= k:
            R[j] = x
```

**Why this works**

The first 100 points are initially stored. After that, the `t`-th point enters the reservoir with probability `100/t`. If it enters, it replaces one of the 100 current reservoir entries uniformly. This maintains equal probability for every data point seen so far while using only `O(100)` memory.


In [ ]:
# Optional reservoir sampling implementation for Problem 2c
import random

class ReservoirSampler:
    def __init__(self, k=100, seed=2026):
        self.k = k
        self.n_seen = 0
        self.sample = []
        self.rng = random.Random(seed)

    def update(self, item):
        self.n_seen += 1
        t = self.n_seen
        if t <= self.k:
            self.sample.append(item)
        else:
            j = self.rng.randint(1, t)
            if j <= self.k:
                self.sample[j - 1] = item
        return self.sample


## Problem 2d [5 pts]

Design an algorithm or method that will estimate, in real-time, and without storing the unique values found so far, the number of unique City ADM3 codes encountered so far. You may answer this problem by means of a narrative (essay), pseudocode, flowchart, and/or illustration. Make sure that you explicitly write down the necessary parameter values.

### Added solution for Problem 2d

## Problem 2d [5 pts] — Estimate number of unique City ADM3 codes without storing uniques

Use **Flajolet-Martin / HyperLogLog-style distinct counting**.

The main idea is to hash every `city_adm3` code and observe how many trailing zero bits appear in the hash. Seeing many trailing zeros is rare. If the maximum observed trailing-zero count is large, then many unique values were probably seen.

A practical version uses multiple registers, as in HyperLogLog, to reduce variance.

**Parameters**

- Number of registers: `m = 1024`, so `b = 10` index bits because `2^10 = 1024`
- Hash function: stable 64-bit hash, such as `xxhash64`
- Register array: `M[0..1023]`, initialized to 0
- HLL bias correction: `alpha_m = 0.7213 / (1 + 1.079/m)`

**Pseudocode**

```text
m = 1024
b = 10
M[0..m-1] = 0

for each incoming record r:
    x = hash64(r.city_adm3)
    register_index = first b bits of x
    remaining_bits = remaining 64-b bits of x
    rank = position of first 1 in remaining_bits
    M[register_index] = max(M[register_index], rank)

estimate = alpha_m * m^2 / sum(2^(-M[j]) for j in 0..m-1)
```

**Memory requirement**

Only `1024` small registers are stored. If each register uses 1 byte, memory is about 1 KB.

**Why this works**

Hash values are approximately uniform. Rare hash patterns, such as many leading or trailing zero bits, become more likely as more distinct values appear. HyperLogLog converts these rare-pattern observations into an estimate of the number of distinct ADM3 codes without storing the codes themselves.


In [ ]:
# Optional approximate distinct count in Spark for Problem 2d.
# Spark's approx_count_distinct uses a HyperLogLog++ style method.
from pyspark.sql import functions as F

adm3_estimate_df = df.agg(
    F.approx_count_distinct("city_adm3", rsd=0.05).alias("estimated_unique_city_adm3")
)


YOUR ANSWER HERE